# Datasets

Six tasks plus a mixture, all in one vocabulary, written as `.npz` under `DATA_ROOT/<dataset>/`.
Run the build cell once (about a minute); it is skipped when the files exist.

In [1]:
import sys, os, json
from pathlib import Path
ROOT = Path.cwd()
while not (ROOT / "backbone").exists():        # works from notebooks/ or models/<s>/
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))
os.chdir(ROOT)
from backbone.paths import DATA_ROOT, RUNS_ROOT, FIG_ROOT
import pandas as pd
pd.set_option("display.width", 200); pd.set_option("display.max_columns", 40)
print("project:", ROOT); print("data:", DATA_ROOT); print("runs:", RUNS_ROOT)

project: /home/claude/work/MoE_Structures_v2
data: /home/claude/work/MoE_Structures_v2/_data
runs: /home/claude/work/MoE_Structures_v2/runs


In [2]:
from data_gen.build_all import main as build
if not (DATA_ROOT / "mix" / "manifest.json").exists():
    sys.argv = ["build_all", "--out", str(DATA_ROOT)]; build()
for d in ["mqar", "fact", "mod", "copy", "rev", "logic", "mix"]:
    m = json.loads((DATA_ROOT / d / "manifest.json").read_text())
    print(f"{d:6s}", {s: (v["n"], v["max_len"], round(v["mean_len"], 1)) for s, v in m["splits"].items()})

mqar   {'train': (200000, 50, 50.0), 'valid': (2000, 50, 50.0), 'test': (4000, 50, 50.0)}
fact   {'train': (200000, 33, 33.0), 'valid': (2000, 33, 33.0), 'test': (4000, 33, 33.0)}
mod    {'train': (200000, 24, 24.0), 'valid': (2000, 24, 24.0), 'test': (4000, 24, 24.0)}
copy   {'train': (200000, 50, 34.0), 'valid': (2000, 50, 34.0), 'test': (4000, 50, 34.2)}
rev    {'train': (200000, 50, 34.0), 'valid': (2000, 50, 33.8), 'test': (4000, 50, 33.8)}
logic  {'train': (121977, 71, 29.9), 'valid': (2000, 65, 30.2), 'test': (4000, 65, 30.3)}
mix    {'train': (199998, 71, 34.1), 'valid': (1998, 62, 34.2), 'test': (3996, 65, 34.3)}


## One example per task (prompt, then the answers expected at the `<ans>` slots)

In [3]:
from backbone.data import Dataset, load_vocab
V = load_vocab()
for d in ["mqar", "fact", "mod", "copy", "rev", "logic"]:
    ds = Dataset(d, "train")
    L = int(ds.lengths[0]); tok, tgt = ds.tokens[0, :L].tolist(), ds.targets[0, :L].tolist()
    print(f"[{d}]", " ".join(V[t] for t in tok))
    print("   answers:", " ".join(V[t] for t in tgt if t >= 0)); print()

[mqar] <t:mqar> n48 n77 n42 n82 n26 n87 n44 n80 n63 n122 n29 n85 n10 n68 n0 n105 n4 n76 n56 n107 n39 n81 n12 n109 n6 n125 n30 n127 n41 n114 n25 n92 <sep> n42 <ans> n41 <ans> n10 <ans> n63 <ans> n12 <ans> n56 <ans> n25 <ans> n48 <ans>
   answers: n82 n114 n68 n122 n109 n107 n92 n77

[fact] <t:fact> e38 e20 r0 <ans> e43 e42 r0 <ans> e86 e29 r0 <ans> e2 e30 r1 <ans> e89 e10 r2 <ans> e39 e19 r3 <ans> e53 e10 r3 <ans> e48 e40 r3 <ans>
   answers: n94 n2 n36 n35 n50 n6 n90 n98

[mod] <t:mod> n89 + n37 = <ans> <sep> n82 * n34 = <ans> <sep> n41 - n52 = <ans> <sep> n38 + n81 = <ans>
   answers: n29 n72 n86 n22

[copy] <t:copy> n4 n39 n35 n49 n20 n60 n3 n15 n0 n62 n37 n18 n9 n6 n6 n24 n49 n3 n29 n7 n60 n63 n9 n15 <sep> <ans> <ans> <ans> <ans> <ans> <ans> <ans> <ans> <ans> <ans> <ans> <ans> <ans> <ans> <ans> <ans> <ans> <ans> <ans> <ans> <ans> <ans> <ans> <ans>
   answers: n4 n39 n35 n49 n20 n60 n3 n15 n0 n62 n37 n18 n9 n6 n6 n24 n49 n3 n29 n7 n60 n63 n9 n15

[rev] <t:rev> n14 n35 n49 n45 n4 n33 